In [1]:
import os
import pandas as pd
import numpy as np
import warnings
import logging
import optuna
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Suppress noisy logs from Prophet/Stan/Optuna so our output stays clean
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Data Ingestion: Daily Data
# ==========================================
# Load preprocessed CSVs for cross-model comparison.
print("Loading Daily Data...")

data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

#  SALES
# Raw data has timestamps, but for daily forecasting we just need one row per date
# with the total units sold for each product
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date']).dt.normalize()
if 'Time' in sales_df.columns:
    sales_df = sales_df.drop(columns=['Time'])
daily_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()

product_cols = [c for c in daily_sales.columns if c != 'Date']

# WEATHER
# Raw weather is hourly, so we aggregate to daily:
#   - Temperature/humidity/etc → daily average
#   - Rain/snow flags → 1 if it happened at ANY hour that day (max)
#   - Precipitation amounts → daily total (sum)
#   - Wind gusts → worst gust of the day (max)
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date'])
if 'weather_code' in weather_df.columns:
    weather_df['is_clear'] = (weather_df['weather_code'] == 0).astype(int)
    weather_df['is_cloudy'] = weather_df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    weather_df['is_rain'] = weather_df['weather_code'].isin(
        [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    weather_df['is_snow'] = weather_df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    weather_df = weather_df.drop(columns=['weather_code'])
weather_df['Date'] = weather_df['Date'].dt.normalize()

weather_agg = {c: 'mean' for c in weather_df.columns if c not in ['Date', 'Time']}
for c in ['is_clear', 'is_cloudy', 'is_rain', 'is_snow']:
    if c in weather_agg:
        weather_agg[c] = 'max'
for c in ['precipitation', 'snowfall']:
    if c in weather_agg:
        weather_agg[c] = 'sum'
for c in ['wind_gusts_10m', 'snow_depth']:
    if c in weather_agg:
        weather_agg[c] = 'max'

daily_weather = weather_df.groupby('Date').agg(weather_agg).reset_index()
if 'Time' in daily_weather.columns:
    daily_weather = daily_weather.drop(columns=['Time'])

#  HOLIDAYS
# One row per date with binary flags (is_bank_holiday, is_school_holiday, etc.)
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date'] = pd.to_datetime(holidays_df['Date']).dt.normalize()
daily_holidays = holidays_df.groupby('Date').max(numeric_only=True).reset_index()

#  EVENTS
# Same format as holidays — binary flags for local events happening on each date
events_df = pd.read_csv(events_path)
events_df['Date'] = pd.to_datetime(events_df['Date']).dt.normalize()
daily_events = events_df.groupby('Date').max(numeric_only=True).reset_index()

# BUILD REGRESSOR LIST
# These are the external variables Prophet will use alongside its own trend/seasonality.
# We combine all weather, holiday, and event columns into one list.
exclude_cols = ['Date', 'Time', 'date', 'Date_Norm', 'Date_Only']
regressors = [c for c in daily_weather.columns if c not in exclude_cols] + \
             [c for c in daily_holidays.columns if c not in exclude_cols] + \
             [c for c in daily_events.columns if c not in exclude_cols]
regressors = list(dict.fromkeys(regressors))  # remove any duplicates

print(f"Products: {product_cols}")
print(f"Regressors ({len(regressors)}): {regressors}")
print(f"Date range: {daily_sales['Date'].min()} to {daily_sales['Date'].max()}")

# --- THE 64 PRODUCTS WE WANT TO FORECAST ---
# This is the exact same list used by XGBoost and LSTM for a fair comparison
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]

/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Loading Daily Data...
Products: ['Almd & Aprct Bar', 'Apl & Blk Smthie', 'Avo & Hal Muffin', 'Avo, Egg & Bacon', 'Avo, Feta & Tom', 'Avocado on Toast', 'BD Curry Roll', 'BLT', 'Bacon', 'Bacon Bap', 'Bacon Egg Brioch', 'Bacon Waffle', 'Baked Beans', 'Baked Beans JP', 'Banana Bread', 'Bean Soldiers', 'Beef Burger', 'Beef Hors Foca', 'Beyond Burger', 'Big Breakfast', 'Black Pudding', 'Blk Forest Syrup', 'Bre Cran Toastie', 'Breakfast Bap', 'Breakfast Hash', 'Breakfast Muffin', 'Breakfast Wrap', 'Brie Bacon Crois', 'Brie Cranb Tstie', 'Brunch Burger', 'Buttd Mushrooms', 'Butter', 'Cheese & Bean JP', 'Cheese JP', 'Cheese Pizza', 'Cheese Sand PnM', 'Cheese Sandwich', 'Cheese Scone', 'Chick Chzo Tstie', 'Chick Flatbread', 'Chick Rice Bowl', 'Chicken Breast', 'Chicken Burger', 'Chicken Club', 'Chicken Goujons', 'Chicken Strips', 'Chicken Waffles', 'Chicken Wrap', 'Chickn Rice Bowl', 'Chilli Carne JP', 'Chips', 'Chorizo', 'Chorizo & Eggs', 'Chse Onion Tstie', 'Chse Tom Toastie', 'Cinnamon Dusti

In [2]:
# Data Ingestion: Demand Clustering
# ==========================================
# Cluster products by demand behavior (volume, volatility, weekend trends) 
# to optimize hyperparameters per group.
print("Building Data-Driven Product Clusters...")

# Only use training data (before Oct 2025) so we don't leak test info into clustering
train_end = pd.to_datetime('2025-10-01')
train_sales = daily_sales[daily_sales['Date'] <= train_end].copy()

# We need temperature to calculate weather sensitivity per product
train_merged = train_sales.merge(daily_weather[['Date', 'apparent_temperature']], on='Date', how='left')
train_merged['dow'] = train_merged['Date'].dt.dayofweek  # Monday=0, Sunday=6

product_features = []
for product in PRODUCTS_TO_FORECAST:
    if product not in train_sales.columns:
        continue
    series = train_sales[product].values.astype(float)
    
    # Average daily sales — tells us if it's a big seller or a niche item
    mean_vol = np.mean(series)
    
    # Coefficient of variation — std / mean — tells us how unpredictable it is
    # A CV of 0.5 means daily sales typically vary by 50% from the average
    std_vol = np.std(series)
    cv = std_vol / mean_vol if mean_vol > 0 else 999.0
    
    # What fraction of days had exactly zero sales
    # High zero-fraction means the product is rarely ordered (hard for Prophet)
    zero_frac = np.mean(series == 0)
    
    # Weekend vs weekday ratio — do people order this more on Sat/Sun?
    # A ratio > 1.0 means it's more popular on weekends (e.g. kids meals, brunch items)
    weekend_mask = train_merged['dow'].isin([5, 6])
    wkend_mean = train_merged.loc[weekend_mask, product].mean()
    wkday_mean = train_merged.loc[~weekend_mask, product].mean()
    weekend_ratio = wkend_mean / wkday_mean if wkday_mean > 0 else 1.0
    
    # Temperature correlation — does this product sell more when it's warm or cold?
    # Positive = sells more in warm weather, negative = sells more in cold weather
    temp_corr = train_merged[[product, 'apparent_temperature']].corr().iloc[0, 1]
    if np.isnan(temp_corr):
        temp_corr = 0.0
    
    product_features.append({
        'product': product,
        'mean_daily_volume': mean_vol,
        'coeff_of_variation': cv,
        'zero_day_fraction': zero_frac,
        'weekend_ratio': weekend_ratio,
        'temp_correlation': temp_corr
    })

features_df = pd.DataFrame(product_features)
print(f"Computed demand-shape features for {len(features_df)} products")

# ==========================================
# RUN KMEANS TO GROUP PRODUCTS INTO CLUSTERS
# ==========================================
# We use 4 clusters — with ~64 products and ~2 years of data, more clusters
# would risk overfitting (each cluster's hero wouldn't represent the group well).
# Fewer clusters would lump together products with genuinely different demand shapes.
N_CLUSTERS = 4

# Standardise features so KMeans doesn't get dominated by mean_daily_volume
# (which can range from 0 to 50+) while weekend_ratio is around 0.5–1.5
feature_cols = ['mean_daily_volume', 'coeff_of_variation', 'zero_day_fraction',
                'weekend_ratio', 'temp_correlation']
X = features_df[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
features_df['cluster_id'] = kmeans.fit_predict(X_scaled)

# For each cluster, pick the highest-volume product as the "hero"
# The hero is the product we'll tune Optuna on — it should be representative
# of the cluster's demand shape, and high volume gives us the cleanest params
PRODUCT_CLUSTER_MAP = {}   # maps each product name -> its cluster number
CLUSTER_HEROES = {}        # maps each cluster number -> the hero product name

for cid in range(N_CLUSTERS):
    cluster_products = features_df[features_df['cluster_id'] == cid]
    hero = cluster_products.sort_values('mean_daily_volume', ascending=False).iloc[0]
    CLUSTER_HEROES[cid] = hero['product']
    for _, row in cluster_products.iterrows():
        PRODUCT_CLUSTER_MAP[row['product']] = cid

#  Print a summary so we can sanity-check the clusters
print(f"\n{'='*60}")
print(f"  CLUSTER SUMMARY ({N_CLUSTERS} clusters)")
print(f"{'='*60}")
for cid in range(N_CLUSTERS):
    members = [p for p, c in PRODUCT_CLUSTER_MAP.items() if c == cid]
    hero = CLUSTER_HEROES[cid]
    hero_row = features_df[features_df['product'] == hero].iloc[0]
    print(f"\n  Cluster {cid} ({len(members)} products)")
    print(f"    Hero product (used for tuning): {hero}")
    print(f"    Hero stats: {hero_row['mean_daily_volume']:.1f} avg/day, "
          f"CV={hero_row['coeff_of_variation']:.2f}, "
          f"Weekend ratio={hero_row['weekend_ratio']:.2f}, "
          f"Temp correlation={hero_row['temp_correlation']:.2f}")
    print(f"    All members: {', '.join(members[:10])}{'...' if len(members) > 10 else ''}")


Building Data-Driven Product Clusters...
Computed demand-shape features for 64 products

  CLUSTER SUMMARY (4 clusters)

  Cluster 0 (37 products)
    Hero product (used for tuning): Bacon Waffle
    Hero stats: 3.2 avg/day, CV=0.77, Weekend ratio=1.80, Temp correlation=0.14
    All members: Avo & Hal Muffin, Avo, Egg & Bacon, Avo, Feta & Tom, Bacon Egg Brioch, Bacon Waffle, Baked Beans JP, Bean Soldiers, Breakfast Hash, Breakfast Muffin, Breakfast Wrap...

  Cluster 1 (24 products)
    Hero product (used for tuning): Bakery
    Hero stats: 52.5 avg/day, CV=0.24, Weekend ratio=1.10, Temp correlation=-0.09
    All members: Avocado on Toast, Bacon, Bacon Bap, Baked Beans, Big Breakfast, Black Pudding, Egg Bacon Brioch, F.Eggs on Toast, Fried Egg, Hash Brown...

  Cluster 2 (2 products)
    Hero product (used for tuning): Festive Stack
    Hero stats: 0.0 avg/day, CV=999.00, Weekend ratio=1.00, Temp correlation=0.00
    All members: Festive Stack, Tuna Mayo Mix

  Cluster 3 (1 products)
 

In [3]:
# Optimization: Cluster-Based Tuning
# ==========================================
# Use Optuna to optimize Prophet parameters (changepoints, seasonality, holidays) 
# for each demand cluster.
print("Running Cluster-Based Optuna Tuning for Prophet...")
print(f"Tuning {N_CLUSTERS} clusters × 20 trials each = {N_CLUSTERS * 20} total Optuna trials")

cluster_best_params = {}

def make_objective(train_data, regressor_list):

    _train = train_data.copy()
    _regs = list(regressor_list)
    
    def objective(trial):
        # These are the hyperparameters Optuna will search over:
        # - changepoint_prior_scale: How flexible the trend line is (higher = more wiggly)
        # - seasonality_prior_scale: How strong seasonal patterns are allowed to be
        # - holidays_prior_scale: How much weight to give holiday effects
        # - changepoint_range: What fraction of the training data can have trend changes
        # - n_changepoints: How many potential trend-change points to place
        # - seasonality_mode: 'additive' (fixed boost) vs 'multiplicative' (proportional boost)
        cps = trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True)
        sps = trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True)
        hps = trial.suggest_float('holidays_prior_scale', 0.01, 10.0, log=True)
        cpr = trial.suggest_float('changepoint_range', 0.7, 0.95)
        n_cp = trial.suggest_int('n_changepoints', 10, 40)
        s_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])

        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            changepoint_prior_scale=cps,
            seasonality_prior_scale=sps,
            holidays_prior_scale=hps,
            changepoint_range=cpr,
            n_changepoints=n_cp,
            seasonality_mode=s_mode
        )
        m.add_country_holidays(country_name='UK')
        for reg in _regs:
            m.add_regressor(reg, standardize=True)

        m.fit(_train)

        # Cross-validation: train on first ~730 days, then slide forward 30 days at a time,
        # predicting 30 days ahead each time. This gives us a realistic estimate of how
        # well these params will perform on unseen data.
        try:
            df_cv = cross_validation(m, initial='730 days', period='30 days', horizon='30 days', parallel='threads')
            df_p = performance_metrics(df_cv, rolling_window=1)
            return df_p['rmse'].values[0]
        except Exception:
            return float('inf')
    
    return objective


#  Run Optuna for each cluster's hero product
for cluster_id, hero_product in CLUSTER_HEROES.items():
    print(f"\n>> Tuning Cluster {cluster_id} (Hero: {hero_product})")
    
    # Build the training dataframe for this hero product
    # Same merge logic as original, just scoped to the hero
    tune_df = daily_sales[['Date', hero_product]].copy()
    tune_df = tune_df.merge(daily_weather, on='Date', how='left')
    tune_df = tune_df.merge(daily_holidays, on='Date', how='left')
    tune_df = tune_df.merge(daily_events, on='Date', how='left')
    
    # Fill missing values: numeric columns get 0, binary flags get clipped to 0/1
    num_cols = tune_df.select_dtypes(include=[np.number]).columns
    tune_df[num_cols] = tune_df[num_cols].fillna(0)
    flag_cols = [c for c in regressors if c.startswith('is_')]
    if flag_cols:
        tune_df[flag_cols] = tune_df[flag_cols].clip(upper=1)
    
    # Prophet expects columns named 'ds' (date) and 'y' (target value)
    tune_df = tune_df.rename(columns={'Date': 'ds', hero_product: 'y'})
    tune_train = tune_df[tune_df['ds'] <= train_end].copy()

    # Create objective with its own data copy and run 20 trials
    objective_fn = make_objective(tune_train, regressors)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_fn, n_trials=20)
    
    # Store the best params for this cluster — all products in this cluster will use these
    cluster_best_params[cluster_id] = study.best_params
    bp = study.best_params
    print(f"  Best CV RMSE: {study.best_value:.4f}")
    print(f"  Seasonality: {bp['seasonality_mode']}, "
          f"CPS: {bp['changepoint_prior_scale']:.4f}, "
          f"SPS: {bp['seasonality_prior_scale']:.4f}, "
          f"HPS: {bp['holidays_prior_scale']:.4f}")

# --- Summary of what Optuna found for each cluster ---
print(f"\n{'='*60}")
print("  TUNING COMPLETE — What each cluster got:")
print(f"{'='*60}")
for cid, params in cluster_best_params.items():
    hero = CLUSTER_HEROES[cid]
    print(f"  Cluster {cid} (tuned on {hero}):")
    print(f"    seasonality_mode={params['seasonality_mode']}, "
          f"changepoint_prior_scale={params['changepoint_prior_scale']:.4f}, "
          f"n_changepoints={params['n_changepoints']}")


Running Cluster-Based Optuna Tuning for Prophet...
Tuning 4 clusters × 20 trials each = 80 total Optuna trials

>> Tuning Cluster 0 (Hero: Bacon Waffle)


13:55:29 - cmdstanpy - INFO - Chain [1] start processing
13:55:29 - cmdstanpy - INFO - Chain [1] done processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] done processing
13:55:32 - cmdstanpy - INFO - Chain [1] done processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] done processing
13:55:32 - cmdstanpy - INFO - Chain [1] done processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] done processing
13:55:32 - cmdstanpy - INFO - Chain [1] start processing
13:55:32 - cmdstanpy - INFO - Chain [1] done processing
13:55:33 - cmdstanpy - INFO - Chain [1] done processing
13:55:33 - cmdstanpy - INFO - Chain [1]

  Best CV RMSE: 2.2615
  Seasonality: additive, CPS: 0.3221, SPS: 0.0213, HPS: 0.0144

>> Tuning Cluster 1 (Hero: Bakery)


13:57:31 - cmdstanpy - INFO - Chain [1] start processing
13:57:31 - cmdstanpy - INFO - Chain [1] done processing
13:57:33 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] done processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] done processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] done processing
13:57:34 - cmdstanpy - INFO - Chain [1] done processing
13:57:34 - cmdstanpy - INFO - Chain [1] done processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [1] start processing
13:57:34 - cmdstanpy - INFO - Chain [

  Best CV RMSE: 11.0436
  Seasonality: additive, CPS: 0.4607, SPS: 0.0133, HPS: 4.9786

>> Tuning Cluster 2 (Hero: Festive Stack)
  Best CV RMSE: 0.0000
  Seasonality: additive, CPS: 0.0015, SPS: 0.1234, HPS: 0.0123

>> Tuning Cluster 3 (Hero: Tuna Melt Panini)


14:00:38 - cmdstanpy - INFO - Chain [1] start processing
14:00:38 - cmdstanpy - INFO - Chain [1] done processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] done processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] done processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] done processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] start processing
14:00:42 - cmdstanpy - INFO - Chain [1] done processing
14:00:42 - cmdstanpy - INFO - Chain [1] done processing
14:00:43 - cmdstanpy - INFO - Chain [1] start processing
14:00:43 - cmdstanpy - INFO - Chain [1] done processing
14:00:43 - cmdstanpy - INFO - Chain [1] done processing
14:00:43 - cmdstanpy - INFO - Chain [1]

  Best CV RMSE: 0.0000
  Seasonality: multiplicative, CPS: 0.0326, SPS: 0.9224, HPS: 1.1823

  TUNING COMPLETE — What each cluster got:
  Cluster 0 (tuned on Bacon Waffle):
    seasonality_mode=additive, changepoint_prior_scale=0.3221, n_changepoints=20
  Cluster 1 (tuned on Bakery):
    seasonality_mode=additive, changepoint_prior_scale=0.4607, n_changepoints=16
  Cluster 2 (tuned on Festive Stack):
    seasonality_mode=additive, changepoint_prior_scale=0.0015, n_changepoints=40
  Cluster 3 (tuned on Tuna Melt Panini):
    seasonality_mode=multiplicative, changepoint_prior_scale=0.0326, n_changepoints=10


In [4]:
# Model Training: Prophet Global Forecast
# ==========================================
# Train individual Prophet models for all products using cluster-optimized parameters.
print("\nTraining Prophet for ALL 74 products using cluster-specific params...")

# November is our blind test set — the model has never seen this data
test_start = pd.to_datetime('2025-11-02')
test_end = pd.to_datetime('2025-11-30')

all_predictions = []

for product in PRODUCTS_TO_FORECAST:
    # Look up which cluster this product belongs to
    # If a product wasn't in the training data (shouldnt happen), default to cluster 0
    cluster_id = PRODUCT_CLUSTER_MAP.get(product, 0)
    params = cluster_best_params[cluster_id]
    
    # Build the feature dataframe for this product
    # Merge sales with weather, holidays, and events (same as original)
    pdf = daily_sales[['Date', product]].copy()
    pdf = pdf.merge(daily_weather, on='Date', how='left')
    pdf = pdf.merge(daily_holidays, on='Date', how='left')
    pdf = pdf.merge(daily_events, on='Date', how='left')
    
    # Clean up: fill NaNs in numeric columns, clip binary flags
    num_cols = pdf.select_dtypes(include=[np.number]).columns
    pdf[num_cols] = pdf[num_cols].fillna(0)
    flag_cols = [c for c in regressors if c.startswith('is_')]
    if flag_cols:
        pdf[flag_cols] = pdf[flag_cols].clip(upper=1)

    # Prophet needs 'ds' and 'y' column names
    prophet_df = pdf.rename(columns={'Date': 'ds', product: 'y'})

    # Split into training (everything up to Oct) and test (November)
    p_train = prophet_df[prophet_df['ds'] <= train_end].copy()
    p_test = prophet_df[(prophet_df['ds'] >= test_start) & (prophet_df['ds'] <= test_end)].copy()

    if len(p_test) == 0:
        continue

    #  Build Prophet model with THIS cluster's tuned hyperparameters
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        holidays_prior_scale=params['holidays_prior_scale'],
        changepoint_range=params['changepoint_range'],
        n_changepoints=params['n_changepoints'],
        seasonality_mode=params['seasonality_mode']
    )
    m.add_country_holidays(country_name='UK')
    for reg in regressors:
        m.add_regressor(reg, standardize=True)

    m.fit(p_train)

    #  Generate predictions for November
    future = p_test[['ds'] + regressors].copy()
    forecast = m.predict(future)

    # Store results — clip negative predictions to 0 (can't sell negative items)
    result = p_test[['ds', 'y']].copy()
    result['Prophet_Prediction'] = np.maximum(forecast['yhat'].values, 0)
    result['Product_Name'] = product
    result['Cluster'] = str(cluster_id)

    #  Build naive baseline (yesterday's sales) for MASE comparison
    # MASE tells us whether our model is better than just using yesterday's number
    full_sales = prophet_df[['ds', 'y']].copy()
    full_sales['sales_1_day_ago'] = full_sales['y'].shift(1)
    result = result.merge(full_sales[['ds', 'sales_1_day_ago']], on='ds', how='left')

    all_predictions.append(result)
    print(f"  {product}: Cluster {cluster_id} ({params['seasonality_mode']}) done")

#  Combine all products into one evaluation dataframe
eval_df = pd.concat(all_predictions, ignore_index=True)
eval_df = eval_df.rename(columns={'ds': 'Date', 'y': 'Sales'})
eval_df['Date_Only'] = eval_df['Date'].dt.date

print(f"\nTotal predictions: {len(eval_df)} rows across {eval_df['Product_Name'].nunique()} products")



Training Prophet for ALL 74 products using cluster-specific params...


14:02:57 - cmdstanpy - INFO - Chain [1] start processing
14:02:58 - cmdstanpy - INFO - Chain [1] done processing


  Avo & Hal Muffin: Cluster 0 (additive) done


14:02:58 - cmdstanpy - INFO - Chain [1] start processing
14:02:58 - cmdstanpy - INFO - Chain [1] done processing


  Avo, Egg & Bacon: Cluster 0 (additive) done


...

  Porridge: Cluster 0 (additive) done


14:03:40 - cmdstanpy - INFO - Chain [1] start processing
14:03:40 - cmdstanpy - INFO - Chain [1] done processing


  Sourdough Toast Bread: Cluster 0 (additive) done


14:03:40 - cmdstanpy - INFO - Chain [1] start processing
14:03:40 - cmdstanpy - INFO - Chain [1] done processing


  Multiseed Toast Bread: Cluster 0 (additive) done

Total predictions: 1856 rows across 64 products


In [5]:
# Evaluation: Model Scorecard
# ==========================================
# North Star Metrics
# ==========================================
# Focus on WAPE for inventory accuracy and MASE for baseline benchmarking. 
# MAE and Bias track unit-level error and prep trends for kitchen staff.

def calculate_north_star_metrics(evaluation_slice):

    if len(evaluation_slice) == 0:
        return {m: 0 for m in ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE']}

    actual = evaluation_slice['Sales'].values.astype(float)
    predicted = evaluation_slice['Prophet_Prediction'].values.astype(float)
    naive = evaluation_slice['sales_1_day_ago'].values.astype(float)

    # WAPE: total absolute error / total actual sales
    total_abs_error = np.sum(np.abs(actual - predicted))
    total_actual = np.sum(np.abs(actual))
    wape = total_abs_error / total_actual if total_actual > 0 else np.nan

    # MASE: our model's MAE / naive baseline's MAE
    mae = mean_absolute_error(actual, predicted)
    naive_mae = mean_absolute_error(actual, naive)
    mase = mae / naive_mae if naive_mae > 0 else np.nan

    # Bias: average (predicted - actual) — positive means we're over-predicting
    bias = np.mean(predicted - actual)

    # Legacy metrics kept for backwards compatibility with dashboards
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)

    nonzero = actual > 0
    mape = mean_absolute_percentage_error(actual[nonzero], predicted[nonzero]) if nonzero.sum() > 0 else np.nan

    abs_err = np.abs(predicted - actual)
    denom = (np.abs(actual) + np.abs(predicted)) / 2.0
    valid = denom > 0
    smape = np.mean(abs_err[valid] / denom[valid]) if valid.sum() > 0 else np.nan

    return {'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': bias,
            'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}


In [6]:
# Evaluation: Per-Product Performance
# ==========================================
# Performance breakdown over 1-day, 1-week, and 1-month horizons.

t1 = pd.to_datetime('2025-11-02').date()
t7 = pd.to_datetime('2025-11-08').date()
t30 = pd.to_datetime('2025-11-30').date()

all_products = sorted(eval_df['Product_Name'].unique())

for product in all_products:
    p_df = eval_df[eval_df['Product_Name'] == product].copy()
    p_df = p_df.sort_values('Date').reset_index(drop=True)

    df_1d = p_df[p_df['Date_Only'] == t1]
    df_1w = p_df[(p_df['Date_Only'] >= t1) & (p_df['Date_Only'] <= t7)]
    df_1m = p_df[(p_df['Date_Only'] >= t1) & (p_df['Date_Only'] <= t30)]

    comparison_df = pd.DataFrame({
        'Metric': ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE'],
        '1-Day (Nov 2)': list(calculate_north_star_metrics(df_1d).values()),
        '1-Week (Nov 2-8)': list(calculate_north_star_metrics(df_1w).values()),
        '1-Month (Nov 2-30)': list(calculate_north_star_metrics(df_1m).values())
    })

    for col in comparison_df.columns[1:]:
        comparison_df[col] = comparison_df[col].apply(
            lambda x: f"{float(x):.4f}" if not np.isnan(x) else "N/A"
        )

    cluster_label = PRODUCT_CLUSTER_MAP.get(product, '?')
    print(f"\n======================================================")
    print(f"  PROPHET DAILY METRICS (BLIND TEST): {product} [Cluster: {cluster_label}]")
    print(f"======================================================")
    print(comparison_df.to_string(index=False))

    # Reality check: show first 10 days actual vs predicted
    p_df['Predicted_Rounded'] = p_df['Prophet_Prediction'].round().astype(int)
    p_df['Mistake_Amount'] = (p_df['Sales'] - p_df['Predicted_Rounded']).abs()

    print(f"\n  REALITY CHECK: {product}")
    print(f"  --------------------------------------------------")
    print(p_df[['Date_Only', 'Sales', 'Predicted_Rounded', 'Mistake_Amount']].head(10).to_string(index=False))



  PROPHET DAILY METRICS (BLIND TEST): Avo & Hal Muffin [Cluster: 0]
Metric 1-Day (Nov 2) 1-Week (Nov 2-8) 1-Month (Nov 2-30)
  WAPE           N/A              N/A                N/A
  MASE           N/A              N/A                N/A
   MAE        0.2535           0.0476             0.0811
  Bias        0.2535           0.0476             0.0811
   MSE        0.0643           0.0095             0.0168
  RMSE        0.2535           0.0976             0.1297
  MAPE           N/A              N/A                N/A
 SMAPE        2.0000           2.0000             2.0000

  REALITY CHECK: Avo & Hal Muffin
  --------------------------------------------------
 Date_Only  Sales  Predicted_Rounded  Mistake_Amount
2025-11-02      0                  0               0
2025-11-03      0                  0               0
2025-11-04      0                  0               0
2025-11-05      0                  0               0
2025-11-06      0                  0               0
2025-11-07  

In [7]:
# Evaluation: Leaderboard & Aggregate Summary
# ==========================================
# Sorted by Bias to highlight operational risk (stock-outs vs waste).
# Aggregated metrics provide a high-level view of model health.

print("\n==================================================")
print("  PER-PRODUCT NORTH STAR METRICS")
print("  Sorted by Bias (most under-predicted first)")
print("==================================================")

per_product_metrics = []
for product_name in all_products:
    product_data = eval_df[eval_df['Product_Name'] == product_name].copy()
    product_month_data = product_data[(product_data['Date_Only'] >= t1) & (product_data['Date_Only'] <= t30)]
    product_metrics = calculate_north_star_metrics(product_month_data)
    product_metrics['Product'] = product_name
    product_metrics['Cluster'] = str(PRODUCT_CLUSTER_MAP.get(product_name, '?'))
    per_product_metrics.append(product_metrics)

product_leaderboard_df = pd.DataFrame(per_product_metrics)
product_leaderboard_df = product_leaderboard_df[['Product', 'Cluster', 'WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']]
product_leaderboard_df = product_leaderboard_df.sort_values('Bias')

for col in ['WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']:
    product_leaderboard_df[col] = product_leaderboard_df[col].apply(
        lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A"
    )

print(product_leaderboard_df.to_string(index=False))

print("\n  HOW TO READ THIS TABLE:")
print("  - WAPE < 0.20  → Good — forecast is within 20% of actual volume")
print("  - MASE < 1.00  → Model beats naive baseline (yesterday's sales as forecast)")
print("  - MASE > 1.00  → Model is WORSE than just using yesterday's number")
print("  - Bias < 0     → Under-predicting → stock-out risk → prep more")
print("  - Bias > 0     → Over-predicting → waste risk → prep less")

#  OVERALL AGGREGATE
all_month = eval_df[(eval_df['Date_Only'] >= t1) & (eval_df['Date_Only'] <= t30)].copy()
overall = calculate_north_star_metrics(all_month)

print("\n==================================================")
print("  OVERALL: Prophet Daily Clustered (All 74 Products)")
print("==================================================")
print(f"  WAPE:  {overall['WAPE']:.4f}  ← % of total inventory wrong")
print(f"  MASE:  {overall['MASE']:.4f}  ← {'BEATING' if overall['MASE'] < 1 else 'LOSING TO'} naive baseline")
print(f"  MAE:   {overall['MAE']:.4f}  ← avg units off per product per day")
print(f"  Bias:  {overall['Bias']:.4f}  ← {'under-predicting (stock-out risk)' if overall['Bias'] < 0 else 'over-predicting (waste risk)'}")
print("  ---")
for m_name in ['RMSE', 'MAPE', 'SMAPE']:
    val = overall[m_name]
    print(f"  {m_name:>6s}: {val:.4f}" if not np.isnan(val) else f"  {m_name:>6s}: N/A")

#  PER-CLUSTER AGGREGATE
# If one cluster is doing much worse than others, you might want to:
#   - Change the number of clusters (N_CLUSTERS)
#   - Give that cluster more Optuna trials
#   - Check if the hero product actually represents the group well
print("\n==================================================")
print("  PER-CLUSTER BREAKDOWN")
print("==================================================")
for cluster_label in sorted(eval_df['Cluster'].unique()):
    cluster_month = all_month[all_month['Cluster'] == cluster_label]
    if len(cluster_month) == 0:
        continue
    cm = calculate_north_star_metrics(cluster_month)
    n_products = cluster_month['Product_Name'].nunique()
    hero = CLUSTER_HEROES.get(int(cluster_label), '?')
    print(f"  Cluster {cluster_label} (tuned on {hero}) — {n_products} products")
    print(f"    WAPE={cm['WAPE']:.4f}, MASE={cm['MASE']:.4f}, MAE={cm['MAE']:.4f}, Bias={cm['Bias']:.4f}")



  PER-PRODUCT NORTH STAR METRICS
  Sorted by Bias (most under-predicted first)
              Product Cluster   WAPE   MASE    MAE    Bias    RMSE
               Bakery       1 0.1565 0.9189 9.8544 -2.7490 11.3547
        Festive Stack       2 1.0000 1.7556 2.7241 -2.7241  3.8011
      Buttd Mushrooms       0 0.6626 0.9939 1.5081 -1.3844  2.0149
        The Breakfast       1 0.1599 0.4368 5.4370 -1.3233  7.0627
          Sausage Bap       1 0.2597 0.7473 3.1437 -1.0517  3.6552
              Egg Bap       0 0.8023 0.7020 1.3556 -1.0365  1.8686
              Tuna JP       0 0.5875 0.7834 1.6208 -0.8996  2.1516
     Bacon Egg Brioch       0 0.6863 0.8074 1.4200 -0.8725  1.7529
        Black Pudding       1 0.6433 0.7213 1.6416 -0.7909  2.1907
      Toasted Teacake       1 0.2469 0.6429 2.1282 -0.7236  2.6131
     Breakfast Muffin       0 0.3741 0.5842 1.4707 -0.6201  1.9924
           Posh Beans       0 0.9517 0.7428 1.0502 -0.4318  1.3192
Multiseed Toast Bread       0 0.5695 0.6133 1.649

In [8]:
# Persistence: SQLite Tracking
# ==========================================
# Use SQLite to maintain a historical record of model performance, 
# allowing for trend analysis in Tableau/PowerBI for the kitchen manager.

import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

# Unique run ID based on timestamp — lets us track every model run
prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_Prophet_Daily_Clustered'

print(f"Saving results for run: {run_id}")

# --- CSV (for quick access) ---
product_leaderboard_df.to_csv('../results/prophet_daily_per_product_results.csv', index=False)
print("  CSVs saved to ../results/")

# --- SQLite (for cross-model comparison) ---
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    # Create tables if this is the first run
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT,
            model_type TEXT,
            target_date TEXT,
            prediction_made_on TEXT,
            product_name TEXT,
            actual_sales REAL,
            predicted_sales REAL
        )
    """)

    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT,
            model_type TEXT,
            product_name TEXT,
            evaluation_horizon TEXT,
            WAPE REAL,
            MASE REAL,
            MAE REAL,
            Bias REAL
        )
    """)

    # Save every individual prediction (for actual-vs-predicted charts)
    predictions_for_db = eval_df[['Date', 'Product_Name', 'Sales', 'Prophet_Prediction']].copy()
    predictions_for_db = predictions_for_db.rename(columns={
        'Date': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'Prophet_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'Prophet_Daily_Clustered'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    # Save aggregated metrics at each time horizon (for "which model wins?" queries)
    evaluation_start = pd.to_datetime('2025-11-02').date()
    evaluation_week_end = pd.to_datetime('2025-11-08').date()
    evaluation_month_end = pd.to_datetime('2025-11-30').date()
    horizon_products = sorted(eval_df['Product_Name'].unique())

    month_slice = eval_df[(eval_df['Date_Only'] >= evaluation_start) & (eval_df['Date_Only'] <= evaluation_month_end)].copy()
    week_slice = month_slice[month_slice['Date_Only'] <= evaluation_week_end].copy()
    day_slice = month_slice[month_slice['Date_Only'] == evaluation_start].copy()
    horizons = {'1-Day': day_slice, '1-Week': week_slice, '1-Month': month_slice}

    metrics_rows = []

    # Per-product metrics at each horizon
    for product_name in horizon_products:
        for horizon_label, horizon_df in horizons.items():
            product_horizon_data = horizon_df[horizon_df['Product_Name'] == product_name] if len(horizon_df) > 0 else horizon_df.iloc[0:0]
            if len(product_horizon_data) > 0:
                m_dict = calculate_north_star_metrics(product_horizon_data)
                metrics_rows.append({
                    'run_id': run_id,
                    'model_type': 'Prophet_Daily_Clustered',
                    'product_name': product_name,
                    'evaluation_horizon': horizon_label,
                    'WAPE': m_dict.get('WAPE', np.nan),
                    'MASE': m_dict.get('MASE', np.nan),
                    'MAE': m_dict.get('MAE', np.nan),
                    'Bias': m_dict.get('Bias', np.nan),
                })

    # Also save an ALL_PRODUCTS aggregate row for each horizon
    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) > 0:
            m_dict = calculate_north_star_metrics(horizon_df)
            metrics_rows.append({
                'run_id': run_id,
                'model_type': 'Prophet_Daily_Clustered',
                'product_name': 'ALL_PRODUCTS',
                'evaluation_horizon': horizon_label,
                'WAPE': m_dict.get('WAPE', np.nan),
                'MASE': m_dict.get('MASE', np.nan),
                'MAE': m_dict.get('MAE', np.nan),
                'Bias': m_dict.get('Bias', np.nan),
            })

    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone! To compare models in the database:")
print("  SELECT model_type, WAPE, MASE, MAE, Bias FROM metrics_summary")
print("    WHERE product_name='ALL_PRODUCTS' AND evaluation_horizon='1-Month'")
print("    ORDER BY WAPE")


Saving results for run: 20260419_1403_Prophet_Daily_Clustered
  CSVs saved to ../results/
  Logged 1856 predictions to predictions_log
  Logged 195 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260419_1403_Prophet_Daily_Clustered

Done! To compare models in the database:
  SELECT model_type, WAPE, MASE, MAE, Bias FROM metrics_summary
    WHERE product_name='ALL_PRODUCTS' AND evaluation_horizon='1-Month'
    ORDER BY WAPE
